# Data Profiling: Yellow - NYC TLC

Este notebook realiza el perfil exploratorio de los datos de los taxis amarillos (Yellow Cab) del proyecto NYC TLC.
Los taxis amarillos operan principalmente en Manhattan y los aeropuertos de Nueva York.
El objetivo es comprender la estructura, distribucion y calidad basica de los datos en crudo.


In [1]:
import os
os.makedirs('images', exist_ok=True)
import sys
sys.path.insert(0, '/home/jovyan/work')
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from src.spark_utils import get_spark, read_parquet
from src.paths import RAW

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

VEHICLE = 'yellow'
RAW_PATH = str(RAW[VEHICLE])
print(f'Raw path: {RAW_PATH}')


Raw path: /home/jovyan/work/data/raw/yellow


## 1. Inicializacion de Spark


In [2]:
spark = get_spark(app_name=f'TLC_Profiling_{VEHICLE.upper()}')

# Leer todos los archivos parquet del directorio raw
df = read_parquet(spark, RAW_PATH)

print(f'Total registros: {df.count():,}')
print(f'Total columnas: {len(df.columns)}')
df.printSchema()


2026-07-18 18:58:12 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
2026-07-18 18:58:18 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/yellow (Robust Mode)
Total registros: 128,202,548
Total columnas: 20
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = t

## 2. Perfil General: Estadisticas Descriptivas


In [3]:
# Estadisticas descriptivas generales
stats = df.describe().toPandas()
print('Estadisticas descriptivas:')
try:
    display(stats)
except NameError:
    print(stats.to_string())

# Conteo de nulos por columna
total_registros = df.count()
null_counts = df.select(
    [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]
).toPandas().T
null_counts.columns = ['nulos']
null_counts['porcentaje'] = (null_counts['nulos'] / total_registros * 100).round(2)

print('\nValores nulos por columna:')
nulos_presentes = null_counts[null_counts['nulos'] > 0].sort_values('porcentaje', ascending=False)
try:
    display(nulos_presentes)
except NameError:
    print(nulos_presentes.to_string())


Estadisticas descriptivas:


,summary,VendorID,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,count,128202548,111190066,128202548,111190066,111190066,128202548,128202548,128202548,128202548,128202548,128202548,128202548,128202548,128202548,128202548,111190066,111190066,48722602
1,mean,1.7931322394621985,1.3332456786202465,5.420458834483844,2.39713233014899,None,163.48070318384,162.7090564845872,1.06574530796377,19.021625775800853,1.3525703114730607,0.4807268720587365,3.194985614482438,0.5461307013961822,0.9630889867337048,27.669647456770374,2.226597432903763,0.14409079224757362,0.5294610907685102
2,stddev,0.5430901807207105,0.8157585255550841,483.82517173189177,11.248101157974906,None,64.97081057116347,70.01359671066236,0.6720421130654306,106.57782631384478,2.034965040428944,0.4802884160149752,4.07135780061412,2.184227385448312,0.2516784906578604,107.51058666288674,0.8775231431511437,0.49924846070558604,0.3585220801850681
3,min,1,0.0,0.0,1.0,N,1,1,0,-2261.2,-39.17,-21.74,-411.0,-148.17,-1.0,-2265.45,-2.5,-1.75,-0.75
4,max,7,9.0,398608.62,99.0,Y,265,265,5,863372.12,10002.5,5243.38,4174.0,1702.88,2.5,863380.37,2.75,6.75,1.75



Valores nulos por columna:


,nulos,porcentaje
cbd_congestion_fee,79479946,62.00
passenger_count,17012482,13.27
RatecodeID,17012482,13.27
store_and_fwd_flag,17012482,13.27
congestion_surcharge,17012482,13.27
airport_fee,17012482,13.27


## 3. Distribucion de Variables Numericas


In [4]:
# â”€â”€ OPTIMIZADO: Histogramas via Spark Bucketing (sin descargar filas crudas) â”€â”€
# En lugar de .sample(5%).toPandas() (45M filas â†’ OOM), calculamos las
# frecuencias directamente en el motor de Spark y solo descargamos los
# conteos resumidos (~40 filas) a Pandas para graficar.

from pyspark.ml.feature import Bucketizer

# Definicion de columnas, rangos y etiquetas
col_cfg = [
    ('trip_distance',   0.0, 60.0,  40, 'Distancia del viaje (millas)'),
    ('fare_amount',     0.0, 200.0, 40, 'Tarifa base (USD)'),
    ('total_amount',    0.0, 200.0, 40, 'Monto total (USD)'),
    ('tip_amount',      0.0, 60.0,  40, 'Propina (USD)'),
    ('passenger_count', 1.0, 7.0,   6,  'Cantidad de pasajeros'),
]

COLOR = 'steelblue'
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
posiciones = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1)]

for idx_col, (col, vmin, vmax, nbins, titulo) in enumerate(col_cfg):
    # Filtrar rango valido en Spark
    df_filt = df.filter(F.col(col).isNotNull() & (F.col(col) >= vmin) & (F.col(col) < vmax))

    # Crear splits para el Bucketizer
    import numpy as _np
    splits = list(_np.linspace(vmin, vmax, nbins + 1).tolist())
    splits[-1] = float('inf')  # capturar el extremo superior

    bkt = Bucketizer(splits=splits, inputCol=col, outputCol='_bucket')
    df_bkt = bkt.transform(df_filt.select(col))

    # Contar en Spark â†’ solo ~nbins filas se descargan a Pandas
    hist_pd = (
        df_bkt.groupBy('_bucket')
              .agg(F.count('*').alias('freq'))
              .orderBy('_bucket')
              .toPandas()
    )

    # Calcular centros de cada bin para el eje X
    step = (vmax - vmin) / nbins
    hist_pd['centro'] = vmin + (hist_pd['_bucket'] + 0.5) * step
    media_val = float(df_filt.select(F.avg(col)).first()[0] or 0)

    fila, columna = posiciones[idx_col]
    ax = fig.add_subplot(gs[fila, columna])
    ax.bar(hist_pd['centro'], hist_pd['freq'], width=step * 0.85,
           color=COLOR, edgecolor='white', alpha=0.85)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor', fontsize=9)
    ax.set_ylabel('Frecuencia', fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.axvline(media_val, color='red', linestyle='--', linewidth=1.2,
               label=f'Media: {media_val:.2f}')
    ax.legend(fontsize=8)

ax_vacio = fig.add_subplot(gs[1, 2])
ax_vacio.set_visible(False)

fig.suptitle(f'Distribucion de Variables Numericas - {VEHICLE.title()} Taxi',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig(f'images/dist_numericas_{VEHICLE}.png', bbox_inches='tight', dpi=120)
plt.show()
print(f'Grafico guardado: dist_numericas_{VEHICLE}.png')


Grafico guardado: dist_numericas_yellow.png


## 4. Distribucion Temporal: Viajes por Ano y Mes


In [5]:
# Extraer ano y mes de la columna de inicio del viaje
df_temporal = df.withColumn('anio', F.year('tpep_pickup_datetime')) \
                .withColumn('mes', F.month('tpep_pickup_datetime')) \
                .withColumn('anio_mes', F.date_format('tpep_pickup_datetime', 'yyyy-MM'))

# Agrupar y contar viajes por anio-mes
conteo_temporal = df_temporal.groupBy('anio_mes') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('anio_mes').isNotNull()) \
    .orderBy('anio_mes') \
    .toPandas()

# Filtrar solo anos razonables
conteo_temporal = conteo_temporal[conteo_temporal['anio_mes'].str[:4].isin(
    [str(y) for y in range(2019, 2026)]
)]

fig, ax = plt.subplots(figsize=(16, 5))
sns.barplot(data=conteo_temporal, x='anio_mes', y='viajes', color='steelblue', ax=ax)
ax.set_title('Viajes por Ano y Mes - Yellow Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Periodo (Ano-Mes)', fontsize=10)
ax.set_ylabel('Cantidad de Viajes', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Rotar etiquetas del eje x
n_etiquetas = len(conteo_temporal)
paso = max(1, n_etiquetas // 24)
etiquetas = [label if i % paso == 0 else '' for i, label in enumerate(conteo_temporal['anio_mes'])]
ax.set_xticklabels(etiquetas, rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('images/temporal_yellow.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: temporal_yellow.png')


Grafico guardado: temporal_yellow.png


## 5. Top 10 Zonas de Mayor Demanda (Pickup)


In [6]:
# Top 10 zonas de recogida por cantidad de viajes
top_zonas = df.groupBy('PULocationID') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('PULocationID').isNotNull()) \
    .orderBy(F.desc('viajes')) \
    .limit(10) \
    .toPandas()

top_zonas['PULocationID'] = top_zonas['PULocationID'].astype(str)
top_zonas = top_zonas.sort_values('viajes', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colores = sns.color_palette('Blues_r', n_colors=len(top_zonas))
bars = ax.barh(top_zonas['PULocationID'], top_zonas['viajes'], color=colores)

# Etiquetas de valor al final de cada barra
for bar_elem in bars:
    ancho = bar_elem.get_width()
    ax.text(ancho * 1.01, bar_elem.get_y() + bar_elem.get_height() / 2,
            f'{ancho:,.0f}', va='center', fontsize=9)

ax.set_title('Top 10 Zonas de Mayor Demanda (Pickup) - Yellow Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Cantidad de Viajes', fontsize=10)
ax.set_ylabel('ID de Zona (PULocationID)', fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('images/top_zonas_yellow.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: top_zonas_yellow.png')


Grafico guardado: top_zonas_yellow.png


## 6. Distribucion de Metodos de Pago


In [7]:
# Mapeo de codigos de pago a etiquetas legibles
mapa_pago = {
    1: 'Tarjeta',
    2: 'Efectivo',
    3: 'Sin cargo',
    4: 'Disputa',
    5: 'Desconocido',
    6: 'Anulado'
}

conteo_pago = df.groupBy('payment_type') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('payment_type').isNotNull()) \
    .orderBy('payment_type') \
    .toPandas()

conteo_pago['etiqueta'] = conteo_pago['payment_type'].map(
    lambda x: mapa_pago.get(int(x), f'Tipo {int(x)}') if x is not None else 'Desconocido'
)

colores_pago = sns.color_palette('Set2', n_colors=len(conteo_pago))

fig, ax = plt.subplots(figsize=(8, 8))
wedges, textos, autotextos = ax.pie(
    conteo_pago['viajes'],
    labels=conteo_pago['etiqueta'],
    autopct='%1.1f%%',
    colors=colores_pago,
    startangle=140,
    pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for texto in autotextos:
    texto.set_fontsize(10)
for texto in textos:
    texto.set_fontsize(11)

ax.set_title('Distribucion de Metodos de Pago - Yellow Taxi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/pago_yellow.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: pago_yellow.png')


Grafico guardado: pago_yellow.png


## 7. Correlacion entre Variables Numericas


In [8]:
# â”€â”€ OPTIMIZADO: Correlacion via pyspark.ml.stat (1 sola lectura del disco) â”€â”€
# El bucle doble original lanzaba 1 scan completo por cada par de columnas.
# VectorAssembler + Correlation.corr calcula la matriz entera en una unica pasada.

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

cols_corr = ['trip_distance', 'fare_amount', 'tip_amount', 'total_amount',
             'tolls_amount', 'passenger_count']

# Filtrar filas con nulos en cualquiera de las columnas de correlacion
df_corr = df.select(cols_corr).dropna()

assembler = VectorAssembler(inputCols=cols_corr, outputCol='features')
df_vec = assembler.transform(df_corr).select('features')

# Una sola pasada distribuida por Spark
corr_matrix = Correlation.corr(df_vec, 'features').head()[0].toArray()

import pandas as _pd
import numpy as _np
matriz_corr = _pd.DataFrame(corr_matrix, index=cols_corr, columns=cols_corr)

etiquetas_corr = {
    'trip_distance': 'Distancia',
    'fare_amount': 'Tarifa',
    'tip_amount': 'Propina',
    'total_amount': 'Total',
    'tolls_amount': 'Peajes',
    'passenger_count': 'Pasajeros'
}
etiquetas = [etiquetas_corr[c] for c in cols_corr]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    matriz_corr.astype(float),
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    xticklabels=etiquetas,
    yticklabels=etiquetas,
    ax=ax,
    linewidths=0.5
)
ax.set_title('Correlacion entre Variables Numericas - Yellow Taxi',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlacion_yellow.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: correlacion_yellow.png')


Grafico guardado: correlacion_yellow.png


## 8. Distribucion Horaria de Viajes


In [9]:
# Extraer hora del inicio del viaje
conteo_hora = df.withColumn('hora', F.hour('tpep_pickup_datetime')) \
    .groupBy('hora') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('hora').isNotNull()) \
    .orderBy('hora') \
    .toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(conteo_hora['hora'], conteo_hora['viajes'],
        marker='o', linewidth=2.2, color='steelblue', markersize=6)
ax.fill_between(conteo_hora['hora'], conteo_hora['viajes'], alpha=0.15, color='steelblue')

ax.set_title('Distribucion Horaria de Viajes - Yellow Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Hora del dia (0-23)', fontsize=10)
ax.set_ylabel('Cantidad de Viajes', fontsize=10)
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(True, linestyle='--', alpha=0.6)

# Marcar hora pico
hora_pico = conteo_hora.loc[conteo_hora['viajes'].idxmax(), 'hora']
viajes_pico = conteo_hora['viajes'].max()
ax.annotate(f'Hora pico: {hora_pico}h',
            xy=(hora_pico, viajes_pico),
            xytext=(hora_pico + 1.5, viajes_pico * 0.95),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

plt.tight_layout()
plt.savefig('images/horario_yellow.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: horario_yellow.png')


Grafico guardado: horario_yellow.png


In [10]:
spark.stop()
print('Profiling completado.')


Profiling completado.
